# FortiOS 8 KVM Lab - With SSH Tunneling & Logging
## Complete lab automation with real-time logging

Control your local KVM infrastructure from Colab with SSH tunneling and comprehensive logging.

---

## STEP 1: Environment Setup

In [ ]:
import os
import sys
import subprocess
import time
from pathlib import Path

print("🚀 Setting up Colab environment...\n")

LAB_DIR = Path('/content/fortios_kvm_lab')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

print(f"📁 Lab directory: {LAB_DIR}")

# Install dependencies
print("\n[*] Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'requests', 'paramiko', 'cryptography', 'flask', 'flask-cors'],
               check=False)

print("✅ Environment ready")

## STEP 2: Download Lab Files

In [ ]:
import urllib.request

print("[*] Downloading lab files...\n")

files = ['kvm_lab_server.py', 'kvm_lab_cli.py', 'ui.html', 'kvm_lab.sh', 'requirements.txt']
base = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"

for f in files:
    try:
        urllib.request.urlretrieve(base + f, f)
        size = os.path.getsize(f) / 1024
        print(f"✓ {f:30} ({size:.1f} KB)")
    except Exception as e:
        print(f"✗ {f:30} Error: {e}")

os.chmod('kvm_lab_cli.py', 0o755)
os.chmod('kvm_lab.sh', 0o755)

print("\n✅ Files downloaded")

## STEP 3: SSH Configuration

In [ ]:
# SSH Configuration
KVM_HOST_IP = "192.168.1.100"      # Your KVM host IP
KVM_HOST_USER = "ubuntu"            # SSH username
KVM_HOST_PORT = 22                  # SSH port

# SSH Authentication - Choose ONE method:
# 1. SSH Key (recommended)
SSH_KEY_PATH = "/root/.ssh/id_rsa"  # Path to private key in Colab
USE_SSH_KEY = True

# 2. SSH Password (less secure)
SSH_PASSWORD = None  # Set if using password auth
USE_PASSWORD = False

# Tunnel Configuration
TUNNEL_LOCAL_PORT = 12443
TUNNEL_REMOTE_HOST = "192.168.122.10"  # FortiOS VM IP
TUNNEL_REMOTE_PORT = 443

print("🔐 SSH Configuration:")
print(f"   KVM Host: {KVM_HOST_USER}@{KVM_HOST_IP}:{KVM_HOST_PORT}")
print(f"   Auth Method: {'SSH Key' if USE_SSH_KEY else 'Password' if USE_PASSWORD else 'Agent'}")
print(f"\n🔗 Tunnel Configuration:")
print(f"   Local:  localhost:{TUNNEL_LOCAL_PORT}")
print(f"   Remote: {TUNNEL_REMOTE_HOST}:{TUNNEL_REMOTE_PORT}")
print(f"\n📝 Edit the configuration above before connecting")

## STEP 4: Start Lab Service

In [ ]:
import subprocess
import time

# Set environment
env = os.environ.copy()
env['KVM_HOST_IP'] = KVM_HOST_IP
env['KVM_HOST_USER'] = KVM_HOST_USER
env['KVM_HOST_PORT'] = str(KVM_HOST_PORT)
env['SSH_KEY_PATH'] = SSH_KEY_PATH
if SSH_PASSWORD:
    env['SSH_PASSWORD'] = SSH_PASSWORD
env['TUNNEL_LOCAL_PORT'] = str(TUNNEL_LOCAL_PORT)
env['TUNNEL_REMOTE_HOST'] = TUNNEL_REMOTE_HOST
env['TUNNEL_REMOTE_PORT'] = str(TUNNEL_REMOTE_PORT)

print("🚀 Starting lab service...\n")

service_proc = subprocess.Popen(
    ['python3', 'kvm_lab_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env,
    cwd=os.getcwd()
)

time.sleep(3)

if service_proc.poll() is None:
    print("✅ Service started")
    print(f"   PID: {service_proc.pid}")
else:
    print("❌ Service failed")
    stdout, stderr = service_proc.communicate()
    print(f"Error: {stderr.decode()}")

%store service_proc

## STEP 5: Connect to KVM

In [ ]:
print("🔌 Connecting to KVM host...\n")

!python3 kvm_lab_cli.py connect

## STEP 6: Setup FortiOS 8

In [ ]:
print("🛠️  Setting up FortiOS 8...\n")

!python3 kvm_lab_cli.py setup

## STEP 7: Start SSH Tunnel

In [ ]:
print("🔗 Starting SSH tunnel...\n")

!python3 kvm_lab_cli.py tunnel-start

## STEP 8: Run Exploitation

In [ ]:
print("🔓 Running exploitation...\n")

!python3 kvm_lab_cli.py exploit --target-ip 127.0.0.1 --target-port 12443

## STEP 9: Post-Exploitation Recon

In [ ]:
print("🔍 Running post-exploitation reconnaissance...\n")

!python3 kvm_lab_cli.py recon --target-ip 127.0.0.1 --target-port 12443

## STEP 10: View Comprehensive Logs

In [ ]:
print("📋 Lab Activity Logs (Last 100 entries)\n")

!python3 kvm_lab_cli.py logs --limit 100

In [ ]:
# Fetch logs via Python for detailed analysis
import requests
import json

print("📊 Log Analysis\n")

response = requests.get('http://localhost:5000/api/logs?limit=50')
logs = response.json().get('logs', [])
total = response.json().get('total', 0)

# Count by level
levels = {}
for log in logs:
    level = log.get('level', 'UNKNOWN')
    levels[level] = levels.get(level, 0) + 1

print(f"Total Logs: {total}")
print(f"\nBreakdown:")
for level, count in sorted(levels.items()):
    icon = {'INFO': '✓', 'ERROR': '✗', 'DEBUG': '→', 'WARN': '⚠'}.get(level, '•')
    print(f"  {icon} {level:6} {count:3} entries")

# Show recent errors if any
errors = [l for l in logs if l.get('level') == 'ERROR']
if errors:
    print(f"\n⚠️  Recent Errors ({len(errors)}):")
    for err in errors[-3:]:
        print(f"   - {err.get('message')}")

## STEP 11: Lab Status

In [ ]:
print("📊 Current Lab Status\n")

!python3 kvm_lab_cli.py status

## STEP 12: Cleanup

In [ ]:
print("🧹 Cleaning up resources...\n")

!python3 kvm_lab_cli.py cleanup

In [ ]:
# Stop service
print("🛑 Stopping lab service...\n")

try:
    %recall service_proc
    if service_proc.poll() is None:
        service_proc.terminate()
        service_proc.wait(timeout=5)
        print("✅ Service stopped")
except:
    !pkill -f kvm_lab_server || true
    print("✅ Service stopped")